In [0]:
from pyspark.sql import functions as F


In [0]:
events_raw = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/",
    header=True,
    inferSchema=True
)

events_raw.printSchema()
events_raw.count()


In [0]:
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events"


In [0]:
events_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)


In [0]:
%fs ls /Volumes/workspace/ecommerce/ecommerce_data/delta/events


In [0]:
events_delta = spark.read.format("delta").load(delta_path)

events_delta.show(5)
events_delta.count()


In [0]:
events_delta.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("events_delta_table")


In [0]:
spark.sql("SHOW TABLES").show()



In [0]:
%sql
CREATE OR REPLACE TABLE events_delta_sql
USING DELTA
AS
SELECT * FROM events_delta_table


In [0]:
%sql
SELECT COUNT(*) FROM events_delta_sql


In [0]:
events_delta.limit(1000).write \
    .format("delta") \
    .mode("append") \
    .save(delta_path)


In [0]:
spark.read.format("delta").load(delta_path).count()



In [0]:
wrong_schema_df = spark.createDataFrame(
    [("a", "b", "c")],
    ["x", "y", "z"]
)


In [0]:
try:
    wrong_schema_df.write \
        .format("delta") \
        .mode("append") \
        .save(delta_path)
except Exception as e:
    print("Schema enforcement working ✅")
    print(e)


In [0]:
parquet_path = "/Volumes/workspace/ecommerce/ecommerce_data/parquet/events"

events_raw.write \
    .mode("overwrite") \
    .parquet(parquet_path)


In [0]:
wrong_schema_df.write \
    .mode("append") \
    .parquet(parquet_path)


In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").show()


In [0]:
old_version = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .load(delta_path)

old_version.count()


In [0]:
spark.read.format("delta").load(delta_path).printSchema()
